# Transfer Bayesian Optimization 使用ガイド

このNotebookは連続変数・単目的・q=1の転移ベイズ最適化を試します。過去タスクのモデルを利用し、ターゲットの観測を逐次追加します。

In [ ]:
import torch
from botorch.models import SingleTaskGP
from robotorchan.optim.transfer.taf_sequential import propose_taf_candidate
from robotorchan.optim.transfer.rgpe_sequential import propose_rgpe_candidate

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0], [1.0]])
def target_objective(x):
    return -((x - 0.75) ** 2)

source_x = torch.linspace(0, 1, 6).unsqueeze(-1)
source_y = -((source_x - 0.7) ** 2)
source_model = SingleTaskGP(source_x, source_y)
target_x = torch.tensor([[0.0], [0.3], [1.0]])
target_y = target_objective(target_x)

## TAFによる逐次候補提案

重みはソース・ターゲットの順で指定します。評価と観測追加は利用者側で行います。

In [ ]:
weights = torch.tensor([0.5, 0.5])
for iteration in range(2):
    candidate = propose_taf_candidate(
        [source_model], target_x, target_y, bounds, weights,
        raw_samples=16, num_restarts=2, seed=iteration,
    )
    observation = target_objective(candidate)
    target_x = torch.cat([target_x, candidate])
    target_y = torch.cat([target_y, observation])
    print(f'TAF iteration={iteration + 1}, x={candidate.item():.4f}, best={target_y.max().item():.4f}')

## RGPEによる候補提案

RGPEはターゲットのLOO予測を得るため複数回GPを学習します。TAFと異なり、重みをランキング損失から推定します。

In [ ]:
candidate, learned_weights = propose_rgpe_candidate(
    [source_model], target_x, target_y, bounds,
    n_bootstrap=8, raw_samples=16, num_restarts=2, seed=0,
)
print('RGPE candidate:', candidate)
print('Source / target weights:', learned_weights)

## 評価と注意点

同一初期観測から手法間の学習曲線を比較するには `examples/benchmarks/transfer_bo.py` を参照してください。負の転移はソースとターゲットの応答面が異なるときに起こり得ます。単一試行の結果から性能の優劣は判断できません。